# 멀티 에이전트 필수 개념 — HandOffs 와 Command

하나의 에이전트로 모든 걸 하기보다, **역할이 다른 여러 에이전트가 협업** 하면 더 잘 푸는 문제가 많다. (예: 조사 담당 + 차트 담당). 이때 핵심이 **HandOff(핸드오프)** — 한 에이전트가 다른 에이전트에게 **제어권을 넘기는 것** 이다.

LangGraph 에서는 **`Command`** 객체로 핸드오프를 구현한다:
- `goto`: 다음에 실행할 노드 이름 (= 제어권을 넘길 대상)
- `update`: 넘기면서 갱신할 State

[basics 복습] 지금까지는 `add_edge`/`add_conditional_edges` 로 흐름을 그래프에 **고정** 했다. `Command` 는 노드가 **자기 반환값으로 직접 다음 행선지를 정한다** — 더 동적이다.

> 이 노트북은 개념 위주라 일부 셀은 LLM/키 없이 동작한다.

## 1. `Command` 로 다음 노드 지정하기

노드가 dict 대신 **`Command(goto=..., update=...)`** 를 반환하면, LangGraph 가 그 노드로 제어를 넘긴다. 아래는 agent_1 → agent_2 → END 로 핸드오프하는 최소 예제 (LLM 불필요).

In [ ]:
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import AIMessage

def agent_1(state: MessagesState):
    msg = AIMessage(content="This is a message from agent 1.")
    # goto 로 'agent_2' 에게 제어권을 넘긴다 (핸드오프)
    return Command(goto="agent_2", update={"messages": [msg]})

def agent_2(state: MessagesState):
    msg = AIMessage(content="This is a message from agent 2.")
    return Command(goto=END, update={"messages": [msg]})

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("agent_1", agent_1)
graph_builder.add_node("agent_2", agent_2)
graph_builder.add_edge(START, "agent_1")
# 주의: agent_1 → agent_2 엣지를 add_edge 로 안 걸었다. Command 의 goto 가 흐름을 만든다.
graph = graph_builder.compile()

In [ ]:
for chunk in graph.stream({"messages": []}, stream_mode="values"):
    print(chunk)

> 위에서 `agent_1 → agent_2` 엣지를 명시적으로 추가하지 않았는데도 흐름이 이어진다. `Command(goto="agent_2")` 가 **런타임에 동적으로** 다음 노드를 결정하기 때문이다.

## 2. 서브그래프 → 부모 그래프로 핸드오프 (`Command.PARENT`)

멀티 에이전트는 종종 **서브그래프**(작은 그래프)를 노드로 품는다. 서브그래프 안에서 **부모 그래프의 노드** 로 제어를 넘기려면 `graph=Command.PARENT` 를 준다.

```python
def node2(state):
    return Command(
        goto="AgentB",                 # 부모 그래프의 노드
        update={"my_state_key": "my_state_value"},
        graph=Command.PARENT,          # '부모 그래프 기준' 으로 goto 해석
    )
```

이 패턴은 06 Hierarchical(팀 단위 계층) 에서 팀 내부 → 상위 supervisor 로 돌아갈 때 쓰인다.

## 3. 도구로서의 핸드오프 (HandOff as Tool)

에이전트가 **도구를 호출하듯이** 다른 에이전트로 넘길 수도 있다. 도구 함수가 `Command` 를 반환하면 그 도구 호출 자체가 핸드오프가 된다. LLM 이 "이건 다른 전문가에게 넘기자" 를 도구 선택으로 표현하는 셈.

```python
from langchain_core.tools import tool
from langgraph.types import Command

@tool
def transfer_to_book():
    """Transfer control to the book agent."""
    return Command(
        goto="book",
        update={"my_state_key": "my_state_value"},
        graph=Command.PARENT,
    )
```

In [ ]:
from langchain_core.tools import tool
from langgraph.types import Command

@tool
def transfer_to_book():
    """Transfer control to the book agent."""
    return Command(goto="book", update={"my_state_key": "my_state_value"}, graph=Command.PARENT)

# 도구로 잘 등록됐는지 확인 (실행 X)
print("tool name:", transfer_to_book.name)
print("description:", transfer_to_book.description)

## 멀티 에이전트 아키텍처 미리보기

이 핸드오프(`Command`)를 어떻게 조합하느냐에 따라 구조가 갈린다:

| 아키텍처 | 구조 | 다음 노트북 |
|---|---|---|
| **Network** | 에이전트들이 서로 직접 핸드오프 (수평) | 02, 03 |
| **Supervisor** | 관리자가 작업자에게 작업 배분 (중앙집중) | 04 |
| **Hierarchical** | 팀(서브그래프)을 상위 supervisor 가 관리 (계층) | 05 |

## 정리
- **핸드오프** = 한 에이전트가 다른 에이전트로 제어권을 넘기는 것
- **`Command(goto=..., update=...)`** = 노드가 반환값으로 다음 노드를 동적으로 지정
- `graph=Command.PARENT` = 서브그래프에서 부모 그래프 노드로 핸드오프
- 도구가 `Command` 를 반환하면 **도구 호출 = 핸드오프** 가 된다

다음: 에이전트들이 서로 직접 넘기는 **Network** 구조 (조사 → 차트).